# 03 — Embedding Generation and FAISS Indexing

Builds the FAISS vector index and explores the embedding space.

Model: `sentence-transformers/all-MiniLM-L6-v2`
- 384 dimensions
- Trained on 1 billion+ sentence pairs
- ~14,000 sentences/second on CPU

In [ ]:
import os
os.chdir('..')  # run from project root so all data/ paths resolve correctly

import sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

## 1. Build the FAISS Index (run once)

In [ ]:
from src.embeddings.indexer import build_index
index, recipe_ids = build_index(config_path='configs/config.yaml')
print(f'Index built: {index.ntotal:,} vectors @ 384 dims')

## 2. Semantic Search Demo

Test that semantically similar queries retrieve relevant recipes.

In [ ]:
from src.embeddings.retriever import RecipeRetriever

retriever = RecipeRetriever(config_path='configs/config.yaml')

test_queries = [
    'high protein post workout meal',
    'diabetic friendly low sugar dinner',
    'gluten free quick lunch',
    'heart healthy low fat soup',
]

for q in test_queries:
    results = retriever.retrieve(q, top_k=3)
    print(f'\nQuery: "{q}"')
    for _, row in results.iterrows():
        flags = [c.replace('is_', '') for c in results.columns if c.startswith('is_') and row[c]]
        print(f'  → {row["name"].title()[:50]:<50} | hybrid={row["hybrid_score"]:.3f} | {flags}')

## 3. Embedding Space Visualization (t-SNE)

Visualize recipe clusters colored by dietary profile.

In [ ]:
# Sample 2000 recipes for visualization
featured = pd.read_parquet('data/processed/recipes_featured.parquet')
sample = featured.sample(2000, random_state=42)

from src.embeddings.encoder import RecipeEncoder
encoder = RecipeEncoder()
embeddings = encoder.encode(sample['recipe_text'].tolist(), show_progress=True)

# Reduce to 2D with t-SNE (via PCA first for speed)
pca_50 = PCA(n_components=50, random_state=42).fit_transform(embeddings)
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(pca_50)

# Color by dietary profile
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
profiles = ['is_diabetic_friendly', 'is_high_protein', 'is_gluten_free', 'is_low_fat']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for ax, flag, color in zip(axes.flatten(), profiles, colors):
    mask = sample[flag].values
    ax.scatter(tsne_2d[~mask, 0], tsne_2d[~mask, 1], c='lightgray', s=5, alpha=0.3)
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1], c=color, s=15, alpha=0.7,
               label=flag.replace('is_', '').replace('_', '-'))
    ax.set_title(flag.replace('is_', '').replace('_', '-').title())
    ax.legend()
    ax.axis('off')

plt.suptitle('Recipe Embedding Space (t-SNE, 2000 sample)', fontsize=14)
plt.tight_layout()
plt.show()